# ADAS Vision — Phase 6b: Fine-tune Drivable-Area Segmentation on Indian Roads

Phase 6a showed pretrained YOLOP (BDD100K = American roads) is excellent on highways/city but **weak on the narrow Himalayan hill road** (frame 15000). Fix: train on Indian roads.

This notebook trains a **lightweight** segmentation model — `LRASPP MobileNetV3-Large` (~3M params, Jetson-friendly) — on **IDD-Lite** (Indian Driving Dataset, IIIT Hyderabad) to predict *drivable area*, then re-runs our six dashcam test frames to compare against YOLOP.

**Before running:**
1. Register (free) at https://idd.insaan.iiit.ac.in/ and download **IDD 20K Lite**
2. Upload the archive to Drive as `MyDrive/adas/idd20k_lite.tar.gz` (or upload the extracted `idd20k_lite` folder to `MyDrive/adas/`)
3. Keep `MyDrive/adas/dashcam.mp4` from Phase 6a
4. Runtime → Change runtime type → **T4 GPU**

> Advisory/research only — never connect any of this to a vehicle's controls.

In [ ]:
# 1) GPU + Drive
!nvidia-smi -L
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 2) Locate / extract IDD-Lite
import os, tarfile, glob

ADAS = "/content/drive/MyDrive/adas"
DATA = "/content/idd20k_lite"

if not os.path.exists(DATA):
    # try an already-extracted folder on Drive first
    drive_folder = os.path.join(ADAS, "idd20k_lite")
    if os.path.isdir(drive_folder):
        DATA = drive_folder
    else:
        archives = glob.glob(os.path.join(ADAS, "idd*lite*.tar.gz")) + \
                   glob.glob(os.path.join(ADAS, "idd*lite*.tgz"))
        assert archives, ("IDD-Lite not found. Download it from https://idd.insaan.iiit.ac.in/ "
                          "and upload as MyDrive/adas/idd20k_lite.tar.gz")
        print("extracting", archives[0], "...")
        with tarfile.open(archives[0]) as t:
            t.extractall("/content")
        # find the extracted root (folder containing leftImg8bit)
        hits = glob.glob("/content/**/leftImg8bit", recursive=True)
        assert hits, "extracted archive but found no leftImg8bit folder inside"
        DATA = os.path.dirname(hits[0])

print("dataset root:", DATA)
print("contents:", os.listdir(DATA))

In [ ]:
# 3) Pair images with labels
# IDD-Lite layout: leftImg8bit/{train,val}/<seq>/<id>_image.jpg
#                  gtFine/{train,val}/<seq>/<id>_label.png
# Label pixel values (level-3 ids): 0 = DRIVABLE, 1 = non-drivable,
# 2 = living things, 3 = vehicles, 4 = roadside objects, 5 = far objects, 6 = sky.

def pairs(split):
    imgs = sorted(glob.glob(os.path.join(DATA, "leftImg8bit", split, "*", "*.jpg")))
    out = []
    for ip in imgs:
        lp = ip.replace("leftImg8bit", "gtFine").replace("_image.jpg", "_label.png")
        if not os.path.exists(lp):   # some releases name labels differently
            lp = ip.replace("leftImg8bit", "gtFine").replace(".jpg", "_label.png")
        if os.path.exists(lp):
            out.append((ip, lp))
    return out

train_pairs, val_pairs = pairs("train"), pairs("val")
print(f"train: {len(train_pairs)} pairs | val: {len(val_pairs)} pairs")
assert train_pairs and val_pairs, "no image/label pairs found — check the dataset layout printed above"

In [ ]:
# 4) Dataset + augmentation (binary target: drivable vs not)
import cv2, numpy as np, random
from torch.utils.data import Dataset, DataLoader

IN_W, IN_H = 320, 224
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD  = np.array([0.229, 0.224, 0.225], np.float32)

class IDDLite(Dataset):
    def __init__(self, pair_list, train=True):
        self.pairs = pair_list
        self.train = train
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        ip, lp = self.pairs[i]
        img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        lbl = cv2.imread(lp, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IN_W, IN_H))
        lbl = cv2.resize(lbl, (IN_W, IN_H), interpolation=cv2.INTER_NEAREST)
        if self.train and random.random() < 0.5:      # horizontal flip
            img, lbl = img[:, ::-1], lbl[:, ::-1]
        if self.train and random.random() < 0.3:      # brightness jitter (shadows!)
            img = np.clip(img.astype(np.float32) * random.uniform(0.6, 1.4), 0, 255)
        x = (img.astype(np.float32) / 255.0 - MEAN) / STD
        x = torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))
        y = torch.from_numpy(np.ascontiguousarray((lbl == 0).astype(np.int64)))  # 0 = drivable
        return x, y

train_dl = DataLoader(IDDLite(train_pairs, True),  batch_size=32, shuffle=True,  num_workers=2)
val_dl   = DataLoader(IDDLite(val_pairs,  False), batch_size=32, shuffle=False, num_workers=2)
print("dataloaders ready")

In [ ]:
# 5) Model: LRASPP MobileNetV3-Large (tiny, fast, Jetson-friendly), 2 classes
import torchvision
from torch import nn

model = torchvision.models.segmentation.lraspp_mobilenet_v3_large(weights="DEFAULT")
model.classifier.low_classifier  = nn.Conv2d(40, 2, 1)
model.classifier.high_classifier = nn.Conv2d(128, 2, 1)
model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"LRASPP MobileNetV3-Large ready — {n_params:.1f}M params")

In [ ]:
# 6) Train (~15-20 min on a T4)
from tqdm import tqdm

EPOCHS = 10
opt = torch.optim.AdamW([
    {"params": model.backbone.parameters(),   "lr": 1e-4},
    {"params": model.classifier.parameters(), "lr": 1e-3},
], weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
lossf = nn.CrossEntropyLoss()

def val_iou():
    model.eval()
    inter = union = 0
    with torch.no_grad():
        for x, y in val_dl:
            p = model(x.to(DEVICE))["out"].argmax(1).cpu()
            inter += ((p == 1) & (y == 1)).sum().item()
            union += ((p == 1) | (y == 1)).sum().item()
    return inter / max(union, 1)

best = 0.0
for ep in range(EPOCHS):
    model.train()
    running = 0.0
    for x, y in tqdm(train_dl, desc=f"epoch {ep+1}/{EPOCHS}"):
        opt.zero_grad()
        out = model(x.to(DEVICE))["out"]
        loss = lossf(out, y.to(DEVICE))
        loss.backward(); opt.step()
        running += loss.item() * x.size(0)
    sched.step()
    iou = val_iou()
    print(f"epoch {ep+1}: loss {running/len(train_pairs):.4f} | val drivable-IoU {iou:.3f}")
    if iou > best:
        best = iou
        torch.save(model.state_dict(), os.path.join(ADAS, "drivable_idd_lraspp.pth"))
print(f"best val IoU {best:.3f} — weights saved to Drive (adas/drivable_idd_lraspp.pth)")

In [ ]:
# 7) THE MOMENT OF TRUTH — our six dashcam test frames.
#    Frame 15000 (Himalayan hill road) is the one YOLOP struggled on.
import matplotlib.pyplot as plt

VIDEO_PATH = os.path.join(ADAS, "dashcam.mp4")
TEST_FRAMES = [2000, 5000, 9000, 12000, 15000, 17000]

model.load_state_dict(torch.load(os.path.join(ADAS, "drivable_idd_lraspp.pth")))
model.eval()

def infer_drivable(frame_bgr, in_w=640, in_h=448):
    """Higher-res inference than training — the net is fully convolutional."""
    h, w = frame_bgr.shape[:2]
    img = cv2.cvtColor(cv2.resize(frame_bgr, (in_w, in_h)), cv2.COLOR_BGR2RGB)
    x = (img.astype(np.float32) / 255.0 - MEAN) / STD
    x = torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        p = model(x)["out"].argmax(1).squeeze().cpu().numpy().astype(np.uint8)
    return cv2.resize(p, (w, h), interpolation=cv2.INTER_NEAREST)

cap = cv2.VideoCapture(VIDEO_PATH)
fig, axes = plt.subplots(len(TEST_FRAMES), 1, figsize=(14, 7 * len(TEST_FRAMES)))
for ax, fi in zip(axes, TEST_FRAMES):
    cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
    ok, frame = cap.read()
    if not ok:
        continue
    da = infer_drivable(frame)
    color = np.zeros_like(frame); color[da == 1] = (0, 180, 0)
    vis = cv2.addWeighted(frame, 1.0, color, 0.45, 0)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"frame {fi} — IDD-fine-tuned drivable area")
    ax.axis("off")
cap.release()
plt.tight_layout(); plt.show()

In [ ]:
# 8) Speed check + ONNX export for future Jetson / OpenVINO deployment
import time

cap = cv2.VideoCapture(VIDEO_PATH); cap.set(cv2.CAP_PROP_POS_FRAMES, 9000)
frames = [cap.read()[1] for _ in range(40)]; cap.release()
for f in frames[:5]: infer_drivable(f)
t0 = time.time()
for f in frames: infer_drivable(f)
print(f"T4 GPU: {len(frames)/(time.time()-t0):.1f} fps at 640x448")

dummy = torch.randn(1, 3, 448, 640).to(DEVICE)
torch.onnx.export(model, dummy, os.path.join(ADAS, "drivable_idd_640x448.onnx"),
                  input_names=["image"], output_names=["seg"], opset_version=12)
print("ONNX exported to Drive (adas/drivable_idd_640x448.onnx)")

## Reading the results

- **Frame 15000 improved?** That's the win condition — IDD training data contains exactly this kind of narrow shaded Indian road.
- **Val IoU ≥ ~0.8** means the model learned the drivable class well on IDD's own validation set.
- The model is ~3M params — small enough for real-time on a Jetson (TensorRT) or near-real-time on a laptop via OpenVINO.

## Next steps
1. Share the frame-15000 result back in the Claude session — it decides deployment.
2. If good: benchmark the ONNX model on the laptop CPU/iGPU (OpenVINO) → wire into `road_detection.py` as a third guidance source (same `LaneDetectionResult` interface).
3. If mixed: train longer / at higher resolution on full IDD (needs Colab Pro or more sessions), or blend YOLOP + this model.